In [2]:
from markov_models import MarkovModel 
from info_rate import compute_info_rate, count_ling_units
from helpers import update_values_in_csv, check_data_availability, load_config, create_minimal_summary
from syllabification import parse_to_phones_and_sylls
import numpy as np
import pickle
from pathlib import Path
import logging
from itertools import product
from joblib import Parallel, delayed
import pandas as pd

# PARALLELIZED VERSION

logging.basicConfig(
    level=logging.INFO,
    format='[%(levelname)s] [%(process)d] %(message)s'
)
logger = logging.getLogger()

def run_pipeline(language, processing_type, text_type, n_values, folder_name, corpus_size):
    folder = Path(folder_name) / language
    config_dict = load_config(language) 
   
    (
        existing_ipa_path,
        phonized_path,
        syllabified_path,
        corpus_size_str,
        is_near_expected,
    ) = parse_to_phones_and_sylls(
        language=language,
        config_dict=config_dict,
        folder=folder,
        corpus_size=corpus_size,
    )

    #input_path = check_data_availability(language, processing_type, config_dict)

    if processing_type == 'words': 
        input_path = existing_ipa_path
    else: 
        input_path = phonized_path if processing_type == 'phones' else syllabified_path

    if not input_path.exists():
        logger.error(f"Input path does not exist: {input_path}")
        return None
    with open(input_path, "rb") as f:
        data = pickle.load(f)
        
    if processing_type == 'words' and text_type == 'within_words':
        return 

    markov_models = {}
    df_rows = []

    for n in n_values:

        # Create and build the Markov model
        model = MarkovModel(n)

        # Build the markov model
        model.build(data, text_type)

        # Compute the conditional entropy (information density)
        info_density = model.compute_conditional_entropy()
        #logger.info(f"Information Density: {info_density:.4f}")

        # Compute the information rate (bits per second)
        info_rate_values, speech_rate_values = compute_info_rate(info_density, processing_type, language)
        #logger.info(f"🧮 {language} | corpus size: {corpus_size_str} | {processing_type} | {text_type} | n={n} | IR: {np.mean(info_rate_values):.4f}")
        
        # Display a table with the computed numbers
        metrics = {
            "ID": info_density,
            "IR": np.mean(info_rate_values),
        }

        df_rows.extend([
            {
                "Language": language,
                "UnitType": processing_type,
                "TextType": text_type,
                "n": n,
                "Metric": metric,
                "Value": round(value, 4),
            }
            for metric, value in metrics.items()
        ])

        # Save the results for a corpus with largest possible size 
        if is_near_expected: 
            update_values_in_csv(language, info_density, n, 'ID', text_type, processing_type)
            update_values_in_csv(language, info_rate_values, n, 'IR', text_type, processing_type)
            update_values_in_csv(language, speech_rate_values, n, 'SR', text_type, processing_type) 

            # Store model for later use 
            markov_models[n] = model
            
            # Save the model to a file
            model.save_model(language, folder, processing_type, text_type, corpus_size_str)
            
    return pd.DataFrame(df_rows) if df_rows else None      


In [6]:
# Create all combinations to process, adjust as needed
languages = ['ENG', 'FRA', 'DEU']
processing_types = ['phones', 'sylls']
text_types = ['within_words', 'across_sentences']
n_values = [1,2,3,4]  
folder_name = "produced_data" 
corpus_size =  5000 # 'max' or specific number (int)

tasks = list(product(languages, processing_types, text_types))

results = Parallel(n_jobs=4, verbose=5)( # Adjust n_jobs based on number of tasks
    delayed(run_pipeline)(lang, proc, txt, n_values, folder_name, corpus_size)
    for lang, proc, txt in tasks
)

# Filter and concatenate DataFrames returned
summary_dfs = [res for res in results if isinstance(res, pd.DataFrame)]
if summary_dfs:
    full_summary = pd.concat(summary_dfs, ignore_index=True)
    create_minimal_summary(full_summary, corpus_size)

success_count = sum(r is not None for r in results)

if success_count == 0:
    logging.error("❌ No valid results. Please check the input data and configurations.")
else:
    logging.info(f"✅ {success_count} tasks completed successfully.")

[Parallel(n_jobs=4)]: Using backend LokyBackend with 4 concurrent workers.


⏩ Skipping tokenization: phonemized and syllabified data already exist for ENG with size 5000.
⏩ Skipping tokenization: phonemized and syllabified data already exist for ENG with size 5000.
⏩ Skipping tokenization: phonemized and syllabified data already exist for ENG with size 5000.


INFO:root:ngram examples: [('m',), ('a',), ('ɪ',), ('p',), ('e',), ('ə',), ('ɹ',), ('ə',), ('n',), ('t',), ('s',), ('w',), ('ʊ',), ('d',), ('ɹ',)]
INFO:root:ngram examples: [('m', 'a'), ('a', 'ɪ'), ('ɪ', 'p'), ('p', 'e'), ('e', 'ə'), ('ə', 'ɹ'), ('ɹ', 'ə'), ('ə', 'n'), ('n', 't'), ('t', 's'), ('s', 'w'), ('w', 'ʊ'), ('ʊ', 'd'), ('d', 'ɹ'), ('ɹ', 'ɪ')]


⏩ Skipping tokenization: phonemized and syllabified data already exist for FRA with size 5000.


INFO:root:ngram examples: [('m', 'a', 'ɪ'), ('a', 'ɪ', 'p'), ('ɪ', 'p', 'e'), ('p', 'e', 'ə'), ('e', 'ə', 'ɹ'), ('ə', 'ɹ', 'ə'), ('ɹ', 'ə', 'n'), ('ə', 'n', 't'), ('n', 't', 's'), ('t', 's', 'w'), ('s', 'w', 'ʊ'), ('w', 'ʊ', 'd'), ('ʊ', 'd', 'ɹ'), ('d', 'ɹ', 'ɪ'), ('ɹ', 'ɪ', 'p')]
INFO:root:ngram examples: [('m', 'a', 'ɪ', 'p'), ('a', 'ɪ', 'p', 'e'), ('ɪ', 'p', 'e', 'ə'), ('p', 'e', 'ə', 'ɹ'), ('e', 'ə', 'ɹ', 'ə'), ('ə', 'ɹ', 'ə', 'n'), ('ɹ', 'ə', 'n', 't'), ('ə', 'n', 't', 's'), ('n', 't', 's', 'w'), ('t', 's', 'w', 'ʊ'), ('s', 'w', 'ʊ', 'd'), ('w', 'ʊ', 'd', 'ɹ'), ('ʊ', 'd', 'ɹ', 'ɪ'), ('d', 'ɹ', 'ɪ', 'p'), ('ɹ', 'ɪ', 'p', 'j')]


...

⏩ Skipping tokenization: phonemized and syllabified data already exist for DEU with size 5000.


INFO:root:ngram examples: [('vas', 'pa', 'siːɾt'), ('pa', 'siːɾt', 'ɪn'), ('siːɾt', 'ɪn', 'dɛɾ'), ('ɪn', 'dɛɾ', 'høː'), ('dɛɾ', 'høː', 'lə'), ('ɪç', 'bɪn', 'nɔø'), ('bɪn', 'nɔø', 'ɡiː'), ('nɔø', 'ɡiː', 'rɪç'), ('ɡiː', 'rɪç', 'ɪç'), ('rɪç', 'ɪç', 'hɑː'), ('ɪç', 'hɑː', 'bə'), ('hɑː', 'bə', 'kaɪ'), ('bə', 'kaɪ', 'nə'), ('kaɪ', 'nə', 'ɑː'), ('nə', 'ɑː', 'nʊŋ')]
INFO:root:ngram examples: [('maɪ', 'peə', 'ɹənts'), ('peə', 'ɹənts', 'wʊd'), ('ɹənts', 'wʊd', 'ɹɪ'), ('wʊd', 'ɹɪ', 'pjuː'), ('ɹɪ', 'pjuː', 'dɪ'), ('pjuː', 'dɪ', 'eɪt'), ('dɪ', 'eɪt', 'maɪ'), ('eɪt', 'maɪ', 'bɹʌ'), ('maɪ', 'bɹʌ', 'ðə'), ('bɹʌ', 'ðə', 'ɪf'), ('ðə', 'ɪf', 'ðeɪ'), ('ɪf', 'ðeɪ', 'ɛ'), ('ðeɪ', 'ɛ', 'və'), ('ɛ', 'və', 'faʊnd'), ('və', 'faʊnd', 'aʊt')]
INFO:root:ngram examples: [('ʒə', 'ne', 'pa', 'fɛ'), ('ne', 'pa', 'fɛ', 'la'), ('pa', 'fɛ', 'la', 'di'), ('fɛ', 'la', 'di', 'fe'), ('la', 'di', 'fe', 'ʁɑ̃s'), ('di', 'fe', 'ʁɑ̃s', 'ɑ̃tʁ'), ('fe', 'ʁɑ̃s', 'ɑ̃tʁ', 'ø'), ('am', 'lɛt', 'a', 'ʒi'), ('lɛt', 'a', 'ʒi', 'kɔm'), ('a',


 Results for corpus size 5000:



[Parallel(n_jobs=4)]: Done  12 out of  12 | elapsed:  5.9min finished


INFO:root:✅ 12 tasks completed successfully.


In [ ]:
from helpers import clean_corpus_size_files
clean_corpus_size_files("produced_data", ["ENG", "FRA", "DEU"], 2000, ['phones', 'sylls'])

In [ ]:
# Create all combinations to process, adjust as needed
languages = ['ENG', 'FRA', 'DEU']
processing_types = ['sylls']
text_types = ['within_words', 'across_sentences']
n_values = [4]  
folder_name = "produced_data_large_corpus" 
corpus_size =  'max' # 'max' or specific number (int)

tasks = list(product(languages, processing_types, text_types))

results = Parallel(n_jobs=4, verbose=5)( # Adjust n_jobs based on number of tasks
    delayed(run_pipeline)(lang, proc, txt, n_values, folder_name, corpus_size)
    for lang, proc, txt in tasks
)

# Filter and concatenate DataFrames returned
summary_dfs = [res for res in results if isinstance(res, pd.DataFrame)]
if summary_dfs:
    full_summary = pd.concat(summary_dfs, ignore_index=True)
    create_minimal_summary(full_summary)

success_count = sum(r is not None for r in results)

if success_count == 0:
    logging.error("❌ No valid results. Please check the input data and configurations.")
else:
    logging.info(f"✅ {success_count} tasks completed successfully.")

In [ ]:
# Create all combinations to process, adjust as needed
languages = ['FRA', 'DEU', 'ENG']
processing_types = ['words']
text_types = ['across_sentences']
n_values = [3,4]  
folder_name = "produced_data_large_corpus"  
corpus_size =  'max' # 'max' or specific number (int)

tasks = list(product(languages, processing_types, text_types))

results = Parallel(n_jobs=4, verbose=10)( # Adjust n_jobs based on number of tasks
    delayed(run_pipeline)(lang, proc, txt, n_values, folder_name, corpus_size)
    for lang, proc, txt in tasks
)

# Filter and concatenate DataFrames returned
summary_dfs = [res for res in results if isinstance(res, pd.DataFrame)]
if summary_dfs:
    full_summary = pd.concat(summary_dfs, ignore_index=True)
    create_minimal_summary(full_summary)

success_count = sum(r is not None for r in results)

if success_count == 0:
    logging.error("❌ No valid results. Please check the input data and configurations.")
else:
    logging.info(f"✅ {success_count} tasks completed successfully.")

In [ ]:
# Create all combinations to process, adjust as needed
languages = ['FRA', 'DEU', 'ENG']
processing_types = ['sylls']
text_types = ['across_sentences']
n_values = [4]  
folder_name = "produced_data_large_corpus"  
corpus_size =  'max' # 'max' or specific number (int)

tasks = list(product(languages, processing_types, text_types))

results = Parallel(n_jobs=4, verbose=5)( # Adjust n_jobs based on number of tasks
    delayed(run_pipeline)(lang, proc, txt, n_values, folder_name, corpus_size)
    for lang, proc, txt in tasks
)

# Filter and concatenate DataFrames returned
summary_dfs = [res for res in results if isinstance(res, pd.DataFrame)]
if summary_dfs:
    full_summary = pd.concat(summary_dfs, ignore_index=True)
    create_minimal_summary(full_summary)

success_count = sum(r is not None for r in results)

if success_count == 0:
    logging.error("❌ No valid results. Please check the input data and configurations.")
else:
    logging.info(f"✅ {success_count} tasks completed successfully.")